## Build an MCP client

```bash
mkdir mcp-client
cd mcp-client

conda create -n TrainingCamp python=3.11 -y
pip install mcp anthropic python-dotenv

# 删除默认生成文件（如果有的话）
del main.py
    
```

## Basic Client Structure

In [1]:
import asyncio, os
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("apikey.env")  # load environment variables from .env

class MCPClient:
    def __init__(self):
        # Initialize session and client objects
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()  ##异步上下文管理器栈，用于自动管理资源关闭（比如关闭 session）
        self.anthropic = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    # methods will go here

## Server Connection Management

根据提供的 server 脚本路径（.py 或 .js），启动该 MCP Server，

建立 双向 STDIO 通信管道，并初始化客户端会话。

连接后，它还会请求服务器端的 “可用工具（tools）” 列表。


In [3]:
async def connect_to_server(self, server_script_path: str):
    """Connect to an MCP server

    Args:
        server_script_path: Path to the server script (.py or .js)
    """
    is_python = server_script_path.endswith('.py')
    is_js = server_script_path.endswith('.js')
    if not (is_python or is_js):
        raise ValueError("Server script must be a .py or .js file")

    command = "python" if is_python else "node"
    server_params = StdioServerParameters(
        command=command,
        args=[server_script_path],
        env=None
    )

    # Client  <--STDIN/STDOUT-->  Server
    # 本质上是一个双工通道：
    stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
    self.stdio, self.write = stdio_transport


    self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))

    await self.session.initialize()

    # List available tools
    response = await self.session.list_tools()
    tools = response.tools
    print("\nConnected to server with tools:", [tool.name for tool in tools])

完成连接后，客户端会调用服务器的 list_tools() 方法。
这通常会返回一个列表，例如：

```py
{
  "tools": [
    {"name": "get_weather"},
    {"name": "forecast_weather"}
  ]
}

```

对用户的自然语言 query 进行 **LLM 调用 → 自动解析 → 工具调用 → 再反馈结果**
相当于一个完整的 「工具增强对话循环」。

In [ ]:
async def process_query(self, query: str) -> str:
    """Process a query using OpenAI and available tools"""
    messages = [
        {
            "role": "user",
            "content": query
        }
    ]

    # 调用 MCP 的标准接口 list_tools()
    response = await self.session.list_tools()
    available_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "describption": tool.description,
            "parameters": tool.inputSchema,
        },
    } for tool in response.tools
]

    # Initial Claude API call
    response = self.openai.messages.create(
        model="gpt-4.1",
        max_tokens=1000,
        messages=messages,
        tools=available_tools
    )

    # Process response and handle tool calls
    final_text = []

    assistant_message_content = []
    for content in response.content:
        
        if content.type == 'text':
            final_text.append(content.text)
            assistant_message_content.append(content)

        elif content.type == 'tool_use':
            tool_name = content.name
            tool_args = content.input

            # Execute tool call
            result = await self.session.call_tool(tool_name, tool_args)
            final_text.append(f"[Calling tool {tool_name} with args {tool_args}]")

            assistant_message_content.append(content)
            messages.append({
                "role": "assistant",
                "content": assistant_message_content
            })
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": content.id,
                        "content": result.content
                    }
                ]
            })

            # Get next response from Claude
            response = self.anthropic.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1000,
                messages=messages,
                tools=available_tools
            )

            final_text.append(response.content[0].text)

    return "\n".join(final_text)

## Interactive Chat Interface
命令行交互的主循环（REPL, Read–Eval–Print Loop）。
当你启动客户端时，它就会保持运行，让你不断输入查询。

In [5]:
async def chat_loop(self):
    """Run an interactive chat loop"""
    print("\nMCP Client Started!")
    print("Type your queries or 'quit' to exit.")

    while True:
        try:
            query = input("\nQuery: ").strip()

            if query.lower() == 'quit':
                break

            response = await self.process_query(query)
            print("\n" + response)

        except Exception as e:
            print(f"\nError: {str(e)}")

async def cleanup(self):
    """Clean up resources"""
    await self.exit_stack.aclose()

In [7]:
import sys

async def main():
    if len(sys.argv) < 2:
        print("Usage: python client.py <path_to_server_script>")
        sys.exit(1)

    client = MCPClient()
    try:
        await client.connect_to_server(sys.argv[1])
        await client.chat_loop()
    finally:
        await client.cleanup()

# if __name__ == "__main__":
#     import sys
#     asyncio.run(main())

```bash
python client.py "C:\Users\hhm18\Desktop\course\state2\mcp\weather\weather.py"
```